## 1. Import necessary libraries

In [3]:
import cv2
import numpy as np
import requests

print("OpenCV version:", cv2.__version__)
print("NumPy version:", np.__version__)
print("Requests version:", requests.__version__)

print("All libraries imported successfully!")

OpenCV version: 5.0.0
NumPy version: 2.5.4
Requests version: 2.34.2
All libraries imported successfully!


## 2. Camera configuration and connection test

In [8]:
import cv2
import json

# Load camera configuration
with open("../config/camera_config.json", "r") as file:
    camera_config = json.load(file)

camera_name = camera_config["camera_name"]
stream_url = camera_config["stream_url"]

# Connect to the IP camera
camera = cv2.VideoCapture(stream_url)

if camera.isOpened():
    print(f"{camera_name} connected successfully!")

    # Read one frame
    success, frame = camera.read()

    if success:
        height, width, channels = frame.shape
        print("Frame captured successfully!")
        print(f"Resolution: {width} x {height}")
        print(f"Frame shape: {frame.shape}")
    else:
        print("Could not read a frame.")
else:
    print("Could not connect to the camera.")

# Release the camera
camera.release()

Phone Camera connected successfully!
Frame captured successfully!
Resolution: 1920 x 1080
Frame shape: (1080, 1920, 3)


## 3. Live Video Streaming

Here, OpenCV is used to continuously capture and display video frames from the connected IP camera. The VideoCapture() function establishes the connection, while read() retrieves each frame. The imshow() function displays the live video and waitKey() checks for keyboard input. Pressing the Esc key stops the stream and releases the camera.

In [9]:
# Connect to the IP camera
camera = cv2.VideoCapture(stream_url)

if not camera.isOpened():
    print("Could not connect to the camera.")

else:
    print("Live streaming started. Press ESC to exit.")

    try:
        while True:
            success, frame = camera.read()

            if not success:
                print("Failed to receive a frame.")
                break

            # Display the live video
            cv2.imshow(camera_name, frame)

            # Check if ESC was pressed
            key = cv2.waitKey(1) & 0xFF

            if key == 27:
                print("Streaming stopped.")
                break

    except KeyboardInterrupt:
        print("Streaming interrupted.")

    finally:
        camera.release()
        cv2.destroyAllWindows()
        print("Camera resources released.")

Live streaming started. Press ESC to exit.
Streaming stopped.
Camera resources released.


## 4. Frame Processing and Information Overlays

In this section, each video frame is processed before being displayed. The frame is resized to a fixed width while maintaining its original aspect ratio. Additional information is displayed on the video, including the camera name, current date and time, resolution and estimated frames per second (FPS). FPS measures how many frames are processed and displayed per second.

In [10]:
import time
from datetime import datetime

# Connect to the IP camera
camera = cv2.VideoCapture(stream_url)

if not camera.isOpened():
    print("Could not connect to the camera.")

else:
    print("Frame processing started. Press ESC to exit.")

    previous_time = time.time()

    try:
        while True:
            success, frame = camera.read()

            if not success:
                print("Failed to receive a frame.")
                break

            # Get original frame dimensions
            height, width, channels = frame.shape

            # Resize while preserving aspect ratio
            display_width = camera_config["display_width"]
            display_height = int(height * display_width / width)

            frame = cv2.resize(frame, (display_width, display_height))

            # Calculate estimated FPS
            current_time = time.time()
            time_difference = current_time - previous_time

            fps = 1 / time_difference if time_difference > 0 else 0
            previous_time = current_time

            # Get current date and time
            timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

            # Get resized resolution
            resolution = f"{display_width} x {display_height}"

            # Add information overlays
            cv2.putText(frame, camera_name, (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

            cv2.putText(frame, timestamp, (10, 60),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

            cv2.putText(frame, f"Resolution: {resolution}", (10, 90),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

            cv2.putText(frame, f"FPS: {fps:.2f}", (10, 120),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

            # Display processed video
            cv2.imshow("IP Camera - Processed Stream", frame)

            # Exit when ESC is pressed
            key = cv2.waitKey(1) & 0xFF

            if key == 27:
                print("Streaming stopped.")
                break

    except KeyboardInterrupt:
        print("Streaming interrupted.")

    finally:
        camera.release()
        cv2.destroyAllWindows()
        print("Camera resources released.")

Frame processing started. Press ESC to exit.
Streaming stopped.
Camera resources released.
